In [363]:
import torch
import numpy as np
import pandas as pd

In [364]:
np.set_printoptions(linewidth = 140)
torch.set_printoptions(linewidth = 140, sci_mode = False, edgeitems = 7)
pd.set_option('display.width', 140)

In [365]:
df = pd.read_csv('train.csv')
print(df.shape)
df

(891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [366]:
modes = df.mode().iloc[0]
df.fillna(modes, inplace = True)

df['Fare'] = np.log(df['Fare'] + 1)
df = pd.get_dummies(df, columns=['Sex', 'Pclass', 'Embarked'], dtype = int)

dummy_cols = ['Sex_male', 'Sex_female', 'Pclass_1', 'Pclass_2', 'Pclass_3', 'Embarked_C', 'Embarked_Q', 'Embarked_S']
indep_cols = ['Age', 'SibSp', 'Parch', 'Fare'] + dummy_cols

In [367]:
t_dep = torch.tensor(df.Survived, dtype = torch.float)
t_ind = torch.tensor(df[indep_cols].values, dtype = torch.float)

In [368]:
t_ind.shape

torch.Size([891, 12])

In [369]:
from fastai.data.transforms import RandomSplitter
trn_split, val_split=RandomSplitter(seed=42)(df)

trn_ind, val_ind = t_ind[trn_split], t_ind[val_split]
trn_dep, val_dep = t_dep[trn_split], t_dep[val_split]
len(trn_ind),len(val_ind)

(713, 178)

In [370]:
def init_params(n_hidden=8):
    layer1 = (torch.rand(t_ind.shape[1], n_hidden) - 0.5) / n_hidden
    layer2 = (torch.rand(n_hidden, 1)) - 0.3
    const = torch.rand(1)[0]
    return layer1.requires_grad_(), layer2.requires_grad_(), const.requires_grad_()

In [371]:
import torch.nn.functional as F

def cal_preds(params, ind_tensor):
    l1, l2, const = params
    res = F.relu(ind_tensor @ l1)
    res = res @ l2 + const
    return torch.sigmoid(res)

In [372]:
def update_params(params, lr):
    for layer in params:
        layer.sub_(layer.grad * lr)
        layer.grad.zero_()

In [373]:
def calc_loss(params, trn_ind, trn_dep): return torch.abs(cal_preds(params, trn_ind) - trn_dep).mean()

In [374]:
def one_epoch(params, lr):
    loss = calc_loss(params, trn_ind, trn_dep)
    loss.backward()
    with torch.no_grad(): update_params(params, lr)
    print(f"{loss:.3f}", end="; ")

In [375]:
def train_model(epochs=30, lr=0.01):
    params = init_params()
    for i in range(epochs): one_epoch(params, lr=lr)
    return params

In [376]:
params = train_model(lr=1.4)

0.571; 0.495; 0.384; 0.384; 0.384; 0.383; 0.383; 0.383; 0.383; 0.382; 0.382; 0.382; 0.382; 0.382; 0.382; 0.382; 0.381; 0.381; 0.381; 0.381; 0.381; 0.381; 0.381; 0.381; 0.381; 0.381; 0.381; 0.380; 0.380; 0.380; 

Deep Learning

In [391]:
t_ind.shape[1]

12

In [395]:
def init_params():
    hiddens = [10, 10]
    sizes = [t_ind.shape[1]] + hiddens + [1]
    layers = [torch.rand(sizes[i], sizes[i+1]) for i in range(len(sizes) - 1)]
    consts = [torch.rand(1)[0] for i in range(len(sizes) - 1)]
    for i in layers + consts: i.requires_grad_()
    return layers, consts

In [396]:
def cal_preds(params, trn_ind):
    layers, consts = params
    res = trn_ind
    for i in range(len(layers)):
        res = (res @ layers[i]) + consts[i]
        if i != len(layers)-1:
            res = F.relu(res)
    
    return torch.sigmoid(res)

In [397]:
def update_params(params, lr):
    layers, consts = params
    for i in range(len(layers) - 1):
        layers[i].sub_(layers[i].grad * lr)
        layers[i].grad.zero_()

In [398]:
train_model(lr=4)

0.621; 0.621; 0.621; 0.621; 0.621; 0.621; 0.621; 0.621; 0.621; 0.621; 0.621; 0.621; 0.621; 0.621; 0.621; 0.621; 0.621; 0.621; 0.621; 0.621; 0.621; 0.621; 0.621; 0.621; 0.621; 0.621; 0.621; 0.621; 0.621; 0.621; 

([tensor([[0.1134, 0.4780, 0.2132, 0.0694, 0.0327, 0.1653, 0.9109, 0.7789, 0.3021, 0.1653],
          [0.1154, 0.2047, 0.2943, 0.8003, 0.6636, 0.3073, 0.8442, 0.1894, 0.0927, 0.6091],
          [0.2243, 0.6422, 0.9236, 0.2635, 0.1511, 0.6882, 0.0484, 0.9461, 0.2639, 0.7417],
          [0.1024, 0.1616, 0.5551, 0.6777, 0.8877, 0.9182, 0.4712, 0.4357, 0.5827, 0.3839],
          [0.5170, 0.0265, 0.0895, 0.8426, 0.2078, 0.8937, 0.5549, 0.6218, 0.9544, 0.2219],
          [0.4211, 0.5886, 0.6708, 0.0134, 0.6777, 0.2051, 0.5657, 0.8383, 0.4502, 0.4842],
          [0.4626, 0.9172, 0.7842, 0.5841, 0.1991, 0.2285, 0.8983, 0.8597, 0.7895, 0.1101],
          [0.8894, 0.4522, 0.8684, 0.9343, 0.5548, 0.2919, 0.9703, 0.2394, 0.8267, 0.0206],
          [0.3427, 0.0641, 0.2757, 0.7123, 0.1417, 0.0676, 0.8575, 0.7669, 0.4945, 0.1475],
          [0.3675, 0.3381, 0.8088, 0.3365, 0.3560, 0.1456, 0.2645, 0.7707, 0.8221, 0.5726],
          [0.3124, 0.9802, 0.5987, 0.3070, 0.3267, 0.4300, 0.7370, 0.6829, 0.862